# XAI Functional Check: Grad-CAM and SHAP

This notebook provides a reproducible functionality check for the two
post-hoc explainability methods used in the project:

- **Grad-CAM**
- **SHAP GradientExplainer**

The purpose of this notebook is to verify that both methods operate
correctly with the trained ConvNeXt V2 Tiny MRI classifier.

This notebook does **not** perform the full explanation-robustness
experiments and does not report final experimental results.


## Methodological Scope

The project investigates whether explanations remain stable when MRI
inputs undergo natural and adversarial perturbations while model
predictions remain unchanged.

The complete study will evaluate explanation robustness under:

- natural noise,
- blur,
- rotation,
- FGSM adversarial perturbations, and
- PGD adversarial perturbations.

Those experiments are outside the scope of this functionality notebook.

Here, one development example is sufficient to verify that Grad-CAM and
SHAP can generate valid spatial explanations from the ConvNeXt V2 model.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

PROJECT_ROOT = Path("/content/drive/MyDrive/MRI_XAI_Robustness")

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))

from xai import (
    GradCAM,
    SHAPExplainer,
    get_convnextv2_gradcam_target,
)


## Required Inputs

This notebook assumes that the model checkpoint and development dataset
have been loaded using the same model and dataset definitions used in
the main-model notebook.

Required runtime objects are:

- `model` — trained ConvNeXt V2 Tiny binary classifier
- `image` — one development MRI stack with shape `[1, 3, 224, 224]`
- `dev_train_dataset` — development-training dataset used to construct
  the SHAP background
- `device` — PyTorch computation device

The SHAP background must be drawn from **training data only**.


## Grad-CAM Functional Check

Grad-CAM is applied to the depthwise convolution in the final
ConvNeXt V2 block.

Because the Stage-1 backbone is frozen, the explanation implementation
enables gradients on an explanation-only copy of the input. This creates
the gradient path required by Grad-CAM without unfreezing or updating
the model parameters.


In [ ]:
model.eval()

target_layer = get_convnextv2_gradcam_target(model)

gradcam = GradCAM(
    model=model,
    target_layer=target_layer,
)

gradcam_map, gradcam_probability = gradcam.generate(image)

print("AD probability:", gradcam_probability)
print("Grad-CAM shape:", tuple(gradcam_map.shape))
print(
    "Grad-CAM range:",
    float(gradcam_map.min()),
    "to",
    float(gradcam_map.max()),
)

gradcam.remove_hooks()


In [ ]:
mri_slice = image[0, 1].detach().cpu().numpy()
cam = gradcam_map.detach().cpu().numpy()

low, high = np.percentile(mri_slice, [1, 99])

mri_display = np.clip(
    (mri_slice - low) / (high - low + 1e-8),
    0,
    1,
)

plt.figure(figsize=(6, 6))

plt.imshow(
    mri_display,
    cmap="gray",
)

plt.imshow(
    cam,
    cmap="jet",
    alpha=0.45,
    vmin=0,
    vmax=1,
)

plt.title(
    "Grad-CAM Functional Check"
)

plt.axis("off")
plt.show()


## SHAP Functional Check

SHAP explanations are generated using `shap.GradientExplainer`.

A small background set is constructed from development-training data
only. Raw signed SHAP values are retained separately for all three MRI
channels.

For visualization and future spatial comparison with Grad-CAM, an
additional two-dimensional attribution-magnitude map is formed by taking
the mean absolute SHAP magnitude across the three adjacent MRI slices.

The magnitude map represents attribution strength and does not by itself
indicate whether an attribution pushes the prediction toward or away
from the AD output.


In [ ]:
background_images = [
    dev_train_dataset[i]["image"]
    for i in range(4)
]

background = torch.stack(
    background_images
).to(device)

shap_explainer = SHAPExplainer(
    model=model,
    background=background,
)

raw_shap, shap_map, shap_probability = (
    shap_explainer.generate(
        image=image,
        nsamples=20,
    )
)

print("AD probability:", shap_probability)
print("Raw SHAP shape:", tuple(raw_shap.shape))
print("2D SHAP map shape:", tuple(shap_map.shape))
print(
    "SHAP map range:",
    float(shap_map.min()),
    "to",
    float(shap_map.max()),
)


In [ ]:
shap_display = shap_map.detach().cpu().numpy()

plt.figure(figsize=(6, 6))

plt.imshow(
    mri_display,
    cmap="gray",
)

plt.imshow(
    shap_display,
    cmap="jet",
    alpha=0.45,
    vmin=0,
    vmax=1,
)

plt.title(
    "SHAP Attribution-Magnitude Functional Check"
)

plt.axis("off")
plt.show()


## Functional Validation

The checks below verify only that both explanation methods return valid,
non-constant spatial maps of the expected dimensions. They are not
measures of explanation robustness.


In [ ]:
assert tuple(gradcam_map.shape) == (224, 224)
assert tuple(raw_shap.shape) == (3, 224, 224)
assert tuple(shap_map.shape) == (224, 224)

assert not torch.isnan(gradcam_map).any()
assert not torch.isnan(shap_map).any()

assert float(gradcam_map.std()) > 0.0
assert float(shap_map.std()) > 0.0

assert abs(
    gradcam_probability - shap_probability
) < 1e-5

print(
    "PASS: Grad-CAM and SHAP explanation "
    "pipelines are functional."
)

print(
    "No robustness conclusions are drawn "
    "from this functionality check."
)


## Next Experimental Stage

The full study will apply the explanation pipeline systematically to the
designated evaluation data. Explanations from unperturbed MRI inputs
will be compared with explanations generated after natural and
adversarial perturbations, subject to the study's prediction-stability
criteria.

That analysis will be conducted separately from this coding-stage
functionality check.
